# 03 · Causal Baselines — Tabular Estimators (No Graph-Aware Exposure)

Run four baseline causal models that do **not** use graph-aware exposure mapping.
These serve as the comparison group for Notebook 04's graph DML.

| ID | Model | Key idea |
|----|-------|----------|
| B1 | OLS (no graph) | Naive regression: `units_sold ~ promo_any + X` |
| B2 | CausalForestDML (no graph) | Heterogeneous treatment effects via causal forest |
| B3 | LinearDML (no graph) | Partially-linear DML, LightGBM nuisance models |
| B4 | LinearDML + unweighted spillover | B3 + 6 pre-computed spillover features as confounders |

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)

cfg = yaml.safe_load((repo_root / "configs" / "causal_config.yaml").read_text())

print("repo root:", repo_root)
print("config loaded ✓")

## 1 — Load panel and embeddings

In [ ]:
panel = pd.read_parquet(repo_root / cfg["data"]["panel_path"])
embeddings = np.load(repo_root / cfg["data"]["embeddings_path"])
pid_idx = np.load(repo_root / cfg["data"]["pid_idx_path"])

print(f"Panel shape: {panel.shape}")
print(f"Embeddings: {embeddings.shape}")
print(f"Weeks: {panel['WEEK_NO'].min()} – {panel['WEEK_NO'].max()}")
panel.head(3)

## 2 — Temporal split

In [ ]:
from src.data.splits import create_temporal_split

splits = create_temporal_split(
    panel,
    train_weeks=tuple(cfg["split"]["train_weeks"]),
    val_weeks=tuple(cfg["split"]["val_weeks"]),
    test_weeks=tuple(cfg["split"]["test_weeks"]),
)

for name, df in splits.items():
    print(f"{name:6s}: {len(df):>9,} rows | weeks {df['WEEK_NO'].min()}-{df['WEEK_NO'].max()}")

train = splits["train"]

## 3 — Run all four baselines

Each model is run individually so you can see per-model timing. Notebook uses fast settings (`n_folds=2`, `n_estimators=50`) on a **1 M-row stratified sample** of the training set (~50 M rows total) to keep runtimes manageable. For publication-quality results use `experiments/baseline_models/run_baselines.py` (`n_folds=5`, `n_estimators=200`, full data).

In [ ]:
import time
from src.models.baselines import (
    run_ols_baseline, run_causal_forest_baseline,
    run_dml_baseline, run_dml_unweighted_spillover,
    results_to_dataframe,
)

# Subsample for notebook iteration — full run is in experiments/baseline_models/run_baselines.py
DEV_SAMPLE_N = 1_000_000
train_sample = train.sample(n=DEV_SAMPLE_N, random_state=cfg["seed"])
print(f"Notebook sample: {len(train_sample):,} rows (from {len(train):,} total training rows)")

DEV = dict(
    panel=train_sample,
    outcome_col=cfg["outcome"]["primary"],
    treatment_col=cfg["treatment"]["primary"],
    embeddings=embeddings,
    emb_pid_idx=pid_idx,
    confidence_level=cfg["dml"]["confidence_level"],
)

baseline_results = []

t0 = time.time(); print("B1  OLS …", end=" ", flush=True)
baseline_results.append(run_ols_baseline(**DEV))
print(f"done ({time.time()-t0:.0f}s)")

t0 = time.time(); print("B2  CausalForest …", end=" ", flush=True)
baseline_results.append(run_causal_forest_baseline(
    **DEV, n_folds=2, n_estimators=48, max_depth=5, seed=cfg["seed"]  # must be divisible by subforest_size=4
))
print(f"done ({time.time()-t0:.0f}s)")

t0 = time.time(); print("B3  LinearDML …", end=" ", flush=True)
baseline_results.append(run_dml_baseline(**DEV, n_folds=2, seed=cfg["seed"]))
print(f"done ({time.time()-t0:.0f}s)")

t0 = time.time(); print("B4  LinearDML + spillover …", end=" ", flush=True)
baseline_results.append(run_dml_unweighted_spillover(**DEV, n_folds=2, seed=cfg["seed"]))
print(f"done ({time.time()-t0:.0f}s)")

print(f"\nFinished {len(baseline_results)} baselines")

## 4 — Compare ATEs and confidence intervals

In [ ]:
summary = results_to_dataframe(baseline_results)
summary

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
y_pos = range(len(summary))
ax.barh(y_pos, summary["ATE"], xerr=[
    summary["ATE"] - summary["CI_lower"],
    summary["CI_upper"] - summary["ATE"],
], color=sns.color_palette("muted"), capsize=4, edgecolor="black", linewidth=0.5)
ax.set_yticks(y_pos)
ax.set_yticklabels(summary["model"])
ax.axvline(0, color="grey", ls="--", lw=0.8)
ax.set_xlabel("ATE (units sold)")
ax.set_title("Baseline ATE Estimates with 95% CI")
plt.tight_layout()
plt.savefig(repo_root / cfg["output"]["figures_dir"] / "baseline_ates.png", dpi=150)
plt.show()

## 5 — CATE distribution (CausalForest)

The causal forest (B2) produces heterogeneous treatment effects. Plot the CATE distribution to see if there is meaningful effect heterogeneity — a prerequisite for graph-based refinement.

In [ ]:
# B2 (causal forest) is the second result
cf_result = baseline_results[1]
cate = cf_result.cate

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(cate, bins=80, edgecolor="white", linewidth=0.3, alpha=0.8)
ax.axvline(cf_result.ate, color="red", ls="--", label=f"ATE = {cf_result.ate:.3f}")
ax.set_xlabel("CATE (units sold)")
ax.set_ylabel("Count")
ax.set_title("CausalForest CATE Distribution (no graph features)")
ax.legend()
plt.tight_layout()
plt.savefig(repo_root / cfg["output"]["figures_dir"] / "cf_cate_distribution.png", dpi=150)
plt.show()
print(f"CATE range: [{cate.min():.3f}, {cate.max():.3f}], std: {cate.std():.3f}")

## 6 — Covariate balance check

Verify that promoted and non-promoted observations are comparable on observed covariates. Large standardised mean differences (|SMD| > 0.1) would indicate confounding that the DML nuisance models need to absorb.

In [ ]:
from src.evaluation.metrics import covariate_balance

balance = covariate_balance(
    train,
    treatment_col=cfg["treatment"]["primary"],
    covariate_cols=["discount_rate", "WEEK_NO", "STORE_ID"],
)
balance

## 7 — Save results

In [ ]:
out_dir = repo_root / cfg["output"]["tables_dir"]
out_dir.mkdir(parents=True, exist_ok=True)

summary.to_csv(out_dir / "baseline_results.csv", index=False)
print(f"Saved to {out_dir / 'baseline_results.csv'}")
summary